## Установка и импорт библиотек

In [10]:
!pip install nltk pymorphy3 gensim fasttext razdel transformers[torch] accelerate

In [11]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

import nltk
import pymorphy3
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
from wordcloud import WordCloud

from razdel import tokenize
import fasttext
import fasttext.util

from datasets import load_dataset
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from collections import Counter

from gensim.models import Word2Vec, FastText

from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

In [12]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Загрузка датасета и его обзор

In [13]:
train_dataset = load_dataset("d0rj/rudetoxifier_data", split="train")
test_dataset = load_dataset("d0rj/rudetoxifier_data", split="test")

README.md:   0%|          | 0.00/989 [00:00<?, ?B/s]

data/train-00000-of-00001-44684fb552b9d8(…):   0%|          | 0.00/15.4M [00:00<?, ?B/s]

data/test-00000-of-00001-3fa069f4ebce7d2(…):   0%|          | 0.00/984k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/163187 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
train = train_dataset.to_pandas()
test = test_dataset.to_pandas()

In [15]:
print(train.shape)
print(test.shape)

(163187, 2)
(10000, 2)


In [16]:
train.head(5)

,text,toxic
0,дворника надо тоже уничтожить!,1.0
1,"моя старшая неделю шипела, не принимала подкид...",0.0
2,полностью с вами согласна!,0.0
3,"хоть ногу вверх, ничего не изменится",0.0
4,а что значит - левого ребенка?,0.0


In [17]:
test.head(5)

,text,toxic
0,долбоебы не иначе,1.0
1,Вечная память героям! Вас будут помнить и чере...,1.0
2,выебать его и его ночяльника и кинуть его жопу...,1.0
3,такую надо ебать,1.0
4,долбоеб старый в хоч импо тент но полиз понюх ...,1.0


In [18]:
train.dtypes

,0
text,object
toxic,float64


In [19]:
train['toxic'] = train['toxic'].astype(int)
train['toxic'] = train['toxic'].astype(int)

## Предобработка

In [20]:
train_copy = train.copy()
test_copy = test.copy()

In [21]:
morph = pymorphy3.MorphAnalyzer()

In [22]:
def preprocess_text(text):
    pattern = re.compile(r'[a-zA-Zа-яА-Я]+', re.UNICODE)
    words = re.findall(pattern, text)

    words = word_tokenize(' '.join(words), language='russian')
    filtered_words = [word for word in words if word.lower() not in stop_words]

    lemmatized_words = [morph.parse(word)[0].normal_form for word in filtered_words]

    return ' '.join(lemmatized_words)

In [24]:
stop_words = stopwords.words("russian")

In [25]:
train_copy['text'] = train_copy['text'].apply(preprocess_text)

In [26]:
train_copy.head(15)

,text,toxic
0,дворник уничтожить,1
1,старший неделя шипеть принимать подкидыш котор...,0
2,полностью вы согласный,0
3,нога вверх измениться,0
4,значит левый ребёнок,0
5,вечер добрый постарелый,0
6,порода,0
7,спасатель просто петух чванливый взять наш пац...,1
8,замечательный юбилей голос чудо заслушаться чт...,0
9,красавец,0


In [27]:
test_copy['text'] = test_copy['text'].apply(preprocess_text)

In [28]:
train_data, val_data = train_test_split(train_copy, test_size=0.10, random_state=42)

In [29]:
print(train_data.shape)
print(val_data.shape)

(146868, 2)
(16319, 2)


## Работа с Transformers и PyTorch

In [30]:
model_name = "cointegrated/rubert-tiny2"

In [31]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [32]:
train_data["label"] = train_data["toxic"].astype(int)
val_data["label"] = val_data["toxic"].astype(int)

In [33]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [34]:
train_loader = DataLoader(TextDataset(train_data, tokenizer), batch_size=64, shuffle=True)
val_loader = DataLoader(TextDataset(val_data, tokenizer), batch_size=64)

In [35]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
)

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [42]:
EPOCHS = 5

In [43]:
history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f'Эпоха {epoch+1}/{EPOCHS} [Train]',
                     leave=False, ncols=100)

    for batch in train_bar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits.squeeze(-1)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(val_loader, desc=f'Эпоха {epoch+1}/{EPOCHS} [Val]',
                   leave=False, ncols=100)

    with torch.no_grad():
      for batch in val_bar:
          input_ids = batch["input_ids"].to(device)
          attention_mask = batch["attention_mask"].to(device)
          labels = batch["labels"].to(device)

          outputs = model(
              input_ids=input_ids,
              attention_mask=attention_mask
          )

          logits = outputs.logits.squeeze(-1)


          loss = criterion(logits, labels)
          val_loss += loss.item()

          probs = torch.sigmoid(logits)
          preds = (probs > 0.5).float()

          val_correct += (preds == labels).sum().item()
          val_total += labels.size(0)

    val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Эпоха {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc * 100:.2f}%')

Эпоха 1/5 [Train]:   0%|                                                   | 0/2295 [00:00<?, ?it/s]

Эпоха 1/5 [Val]:   0%|                                                      | 0/255 [00:00<?, ?it/s]

Эпоха  1/5 | Train Loss: 0.1933 | Val Loss: 0.1764 | Val Acc: 92.74%


Эпоха 2/5 [Train]:   0%|                                                   | 0/2295 [00:00<?, ?it/s]

Эпоха 2/5 [Val]:   0%|                                                      | 0/255 [00:00<?, ?it/s]

...

Эпоха 4/5 [Val]:   0%|                                                      | 0/255 [00:00<?, ?it/s]

Эпоха  4/5 | Train Loss: 0.1499 | Val Loss: 0.1839 | Val Acc: 93.82%


Эпоха 5/5 [Train]:   0%|                                                   | 0/2295 [00:00<?, ?it/s]

Эпоха 5/5 [Val]:   0%|                                                      | 0/255 [00:00<?, ?it/s]

Эпоха  5/5 | Train Loss: 0.1542 | Val Loss: 0.2223 | Val Acc: 92.53%
